# Text, tabular data, and recommendations: three representation problems

The input boundary changes across modalities, but the modeling discipline stays the same: define a baseline, choose a representation, train on observed evidence, and state what the model cannot know. The examples below are small by design; each one exposes the representation rather than hiding it behind a library call.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import torch
from torch import nn
from sklearn.metrics import accuracy_score, mean_squared_error
sns.set_theme(style='whitegrid',context='notebook'); torch.manual_seed(2); np.random.seed(2)

## 1. Text: a conditional table before an embedding model

For a character-level language model, the simplest baseline stores counts `C[a,b]` and estimates

`P(next=b | current=a) = (C[a,b] + α) / (Σ_b C[a,b] + αV)`.

Smoothing prevents an unseen transition from receiving probability zero. An embedding model replaces the table with learned vectors and a neural scoring function, but the prediction target is the same.

In [ ]:
text='the cat sat on the mat. the cat slept.'; vocab=sorted(set(text)); to_id={c:i for i,c in enumerate(vocab)}
counts=np.ones((len(vocab),len(vocab)))
for a,b in zip(text,text[1:]): counts[to_id[a],to_id[b]]+=1
probs=counts/counts.sum(axis=1,keepdims=True)
print('vocabulary size:',len(vocab),'row sums:',np.round(probs.sum(1)[:4],3))
context='t'; next_ids=np.argsort(probs[to_id[context]])[::-1][:5]
print('most likely after t:',[(vocab[i],round(probs[to_id[context],i],3)) for i in next_ids])

## 2. Categorical tabular features need learned representations

A category code is an identifier, not a measurement. If `desktop=0` and `mobile=1`, the difference between them has no numerical meaning. An embedding learns a vector for each category. Continuous values are normalized separately and then concatenated with those vectors.

In [ ]:
rows=pd.DataFrame({'device':['mobile','desktop','mobile','tablet','desktop','mobile','tablet','desktop'],'region':['south','north','north','south','west','west','north','west'],'hours':[1.2,4.1,2.4,3.0,5.2,.8,2.8,4.7],'converted':[0,1,1,0,1,0,1,1]})
enc={c:{v:i for i,v in enumerate(sorted(rows[c].unique()))} for c in ['device','region']}
cat=torch.tensor([[enc[c][getattr(r,c)] for c in ['device','region']] for r in rows.itertuples()])
continuous=torch.tensor(rows[['hours']].to_numpy(),dtype=torch.float32)
embs=[nn.Embedding(len(enc[c]),3) for c in ['device','region']]
representation=torch.cat([emb(cat[:,j]) for j,emb in enumerate(embs)]+[continuous],1)
print(rows); print('representation shape:',tuple(representation.shape))

## 3. A tabular baseline and a neural model

A tree baseline is often strong on small tabular data because it can split a category or threshold directly. The neural model below is not automatically superior; it is useful for showing the role of embeddings and for cases where the representation will later be shared with another model.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
X=rows[['device','region','hours']]; y=rows.converted
prep=ColumnTransformer([('cat',OneHotEncoder(handle_unknown='ignore'),['device','region']),('num',StandardScaler(),['hours'])])
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,stratify=y,random_state=2)
Xtr=prep.fit_transform(X_train); Xte=prep.transform(X_test)
tree=RandomForestClassifier(n_estimators=100,random_state=2).fit(Xtr,y_train)
print('tree test accuracy:',accuracy_score(y_test,tree.predict(Xte)))

## 4. Collaborative filtering and cold starts

For an observed user-item pair `(u,i)`, a matrix-factorization model predicts

`r_hat(u,i) = μ + b_u + b_i + p_u · q_i`.

Only observed ratings should contribute to the loss. A new user has no learned `p_u`; the model must fall back to a global or side-information-based estimate. Treating every missing matrix entry as a negative rating produces a different and usually misleading objective.

In [ ]:
ratings=torch.tensor([[5.,0.,3.,0.],[0.,4.,0.,2.],[1.,0.,4.,0.]])
users,items=torch.where(ratings>0); values=ratings[users,items]
P=nn.Embedding(3,2); Q=nn.Embedding(4,2); bu=nn.Embedding(3,1); bi=nn.Embedding(4,1); mu=nn.Parameter(torch.tensor(3.0))
opt=torch.optim.Adam(list(P.parameters())+list(Q.parameters())+list(bu.parameters())+list(bi.parameters())+[mu],lr=.03)
for _ in range(800):
    pred=mu+bu(users).squeeze()+bi(items).squeeze()+(P(users)*Q(items)).sum(1)
    loss=((pred-values)**2).mean(); opt.zero_grad(); loss.backward(); opt.step()
print('observed RMSE:',float(loss.sqrt()),'cold-start fallback:',float(mu.detach()))

### Limits

These tiny examples establish representations and baselines. They do not establish generalization: the tabular sample is too small for a reliable accuracy estimate, and the recommender has no held-out interactions. A real project needs a time-aware split or user/item holdout, leakage checks, and an error analysis.